In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cài thêm nếu chưa có
# !pip install pandas openpyxl scikit-learn

import re
import pandas as pd
from sklearn.model_selection import train_test_split



In [ ]:
df = pd.read_excel("/content/drive/MyDrive/Nguyễn Thị Sương Anh /Do_an_co_so/DACS/data/TONG_HOP .xlsx", sheet_name="db")

# Drop duplicate header rows (rows where sentiment == "sentiment")
df = df[df["sentiment"] != "sentiment"].reset_index(drop=True)

print(f"Dataset loaded: {len(df)} rows")
print(df["sentiment"].value_counts())

Dataset loaded: 28091 rows
sentiment
neutral     10593
positive     9778
negative     7720
Name: count, dtype: int64


In [ ]:
tc_df = pd.read_excel("/content/drive/MyDrive/Nguyễn Thị Sương Anh /Do_an_co_so/DACS/data/teencode_final .xlsx", sheet_name="Từ điển teencode")

# Loại bỏ các dòng null
tc_df = tc_df.dropna(subset=["Teencode", "Dạng chuẩn hóa"])

# Build dictionary: key = teencode (str, stripped), value = chuẩn hóa (str, stripped)
dictionary = {
    str(row["Teencode"]).strip(): str(row["Dạng chuẩn hóa"]).strip()
    for _, row in tc_df.iterrows()
}

print(f"\nTeen dictionary loaded: {len(dictionary)} entries")
print("Ví dụ:", list(dictionary.items())[:5])


Teen dictionary loaded: 776 entries
Ví dụ: [('huhu', 'khóc'), ('hi', 'xin chào / cười'), ('vớii', 'với ơi (nhấn)'), ('ha', 'hả / ừ ha'), ('iu', 'yêu / thân mật')]


In [ ]:
# Dictionary emoji → token cảm xúc
EMOJI_MAP = {
    #Positive
    '🙂': 'Positive', '😀': 'Positive', '😄': 'Positive', '😆': 'Positive',
    '😅': 'Positive', '😂': 'Positive', '😊': 'Positive', '😌': 'Positive',
    '😉': 'Positive', '😏': 'Positive', '😍': 'Positive', '🙃': 'Positive',
    '😺': 'Positive', '🎃': 'Positive', '💩': 'Positive', '😎': 'Positive',
    '😋': 'Positive', '😜': 'Positive', '😝': 'Positive', '😛': 'Positive',
    '😈': 'Positive', '😇': 'Positive', '😸': 'Positive', '😹': 'Positive',
    '😼': 'Positive', '🌜': 'Positive', '🌛': 'Positive', '🌚': 'Positive',
    '🌝': 'Positive', '🌞': 'Positive', '👍': 'Positive', '👌': 'Positive',
    '✌': 'Positive', '🙌': 'Positive', '💯': 'Positive', '🙋': 'Positive',
    '✋': 'Positive', '✅': 'Positive', '✔': 'Positive', '👏': 'Positive',
    '💪': 'Positive', '🙏': 'Positive', '☀': 'Positive', '👉': 'Positive',
    '🏃': 'Positive', '☝': 'Positive',
    # thêm các icon cảm xúc tích cực phổ biến khác
    '🥰': 'Positive', '🤩': 'Positive', '😘': 'Positive', '🤗': 'Positive',
    '😁': 'Positive', '🤭': 'Positive', '🥳': 'Positive', '🎉': 'Positive',
    '🎊': 'Positive', '🌟': 'Positive', '⭐': 'Positive', '💖': 'Positive',
    '💗': 'Positive', '💓': 'Positive', '💞': 'Positive', '💕': 'Positive',
    '❤️': 'Positive', '🧡': 'Positive', '💛': 'Positive', '💚': 'Positive',
    '💙': 'Positive', '💜': 'Positive', '🤝': 'Positive', '🙆': 'Positive',
    '💐': 'Positive', '🌹': 'Positive', '🌺': 'Positive', '🌸': 'Positive',
    '🍀': 'Positive', '🎶': 'Positive', '🎵': 'Positive', '🔥': 'Positive',
    '✨': 'Positive', '🌈': 'Positive', '😃': 'Positive', '🤣': 'Positive',
    #Negative
    '🙁': 'Negative', '☹': 'Negative', '😞': 'Negative', '😖': 'Negative',
    '😔': 'Negative', '😓': 'Negative', '😢': 'Negative', '😭': 'Negative',
    '😟': 'Negative', '🙎': 'Negative', '😿': 'Negative', '😰': 'Negative',
    '😱': 'Negative', '🙀': 'Negative', '😧': 'Negative', '😨': 'Negative',
    # thêm các icon cảm xúc tiêu cực phổ biến khác
    '😤': 'Negative', '😠': 'Negative', '😡': 'Negative', '🤬': 'Negative',
    '🤢': 'Negative', '🤮': 'Negative', '😩': 'Negative', '😫': 'Negative',
    '😥': 'Negative', '😪': 'Negative', '💔': 'Negative', '👎': 'Negative',
    '🤦': 'Negative', '🤦‍♀️': 'Negative', '🤦‍♂️': 'Negative',
    '😒': 'Negative', '🙅': 'Negative', '🙅‍♀️': 'Negative',
    '😑': 'Negative', '😬': 'Negative', '🥺': 'Negative',
    # Neutral
    '🙄': 'Neutral', '💥': 'Neutral', '😲': 'Neutral', '😳': 'Neutral',
    # thêm các icon trung lập phổ biến khác
    '😐': 'Neutral', '😶': 'Neutral', '🤔': 'Neutral', '🤷': 'Neutral',
    '🤷‍♀️': 'Neutral', '🤷‍♂️': 'Neutral', '😴': 'Neutral',
    '🧐': 'Neutral', '😯': 'Neutral', '🙃': 'Neutral',
}

print(f"📋 Tổng số emoji trong bảng: {len(EMOJI_MAP)}")
positive_count = sum(1 for v in EMOJI_MAP.values() if v == 'Positive')
negative_count = sum(1 for v in EMOJI_MAP.values() if v == 'Negative')
neutral_count  = sum(1 for v in EMOJI_MAP.values() if v == 'Neutral')
print(f"   😊 Positive : {positive_count}")
print(f"   😢 Negative : {negative_count}")
print(f"   😐 Neutral  : {neutral_count}")
# Regex xóa toàn bộ emoji còn lại sau khi đã map
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"   # emoticons
    "\U0001F300-\U0001F5FF"   # symbols & pictographs
    "\U0001F680-\U0001F6FF"   # transport & map
    "\U0001F1E0-\U0001F1FF"   # flags
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "\U0001F900-\U0001F9FF"   # supplemental symbols (thêm mới)
    "\U0001FA00-\U0001FA6F"   # chess, medical
    "\U0001FA70-\U0001FAFF"   # food, drink, objects
    "\U00002300-\U000023FF"   # misc technical
    "]+",
    flags=re.UNICODE
)

# Các pattern Facebook cần xóa
FACEBOOK_PATTERNS = [
    r"added.*?photo", r"added.*?photos",
    r"is.*?post",     r"photos.*?post",
    r"from.*?post",   r"shared.*?group",
    r"shared.*?post", r"shared.*?video",
    r"is.*?motivated",r"is.*?with",
]

def Preprocess(string: str) -> str:
    """
    Pipeline tiền xử lý văn bản tiếng Việt:
      1. Ép kiểu str
      2. Thay emoji có nghĩa → token Positive/Negative/Neutral
      3. Xóa emoji còn lại
      4. Rút gọn ký tự lặp (vd: đẹppppp → đẹp)
      5. Viết thường
      6. Xóa thẻ HTML
      7. Xóa ký tự đặc biệt
      8. Thay số → token 'number'
      9. Xóa pattern Facebook
     10. Chuẩn hóa khoảng trắng
    """
    if not isinstance(string, str):
        string = str(string)

    #   Thay emoji có nghĩa TRƯỚC khi xóa
    for emoji, token in EMOJI_MAP.items():
        string = string.replace(emoji, f" {token} ")

    #   Xóa phần emoji còn lại
    string = EMOJI_PATTERN.sub("", string)

    #   Rút gọn ký tự lặp: đẹppppp → đẹp
    string = re.sub(r"([A-Za-z])\1+", lambda m: m.group(1), string, flags=re.IGNORECASE)

    #   Viết thường
    string = string.lower()

    #   Xóa thẻ HTML
    string = re.sub(r"<.*?>", "", string)

    #   Xóa ký tự đặc biệt (giữ chữ Việt, Latin, khoảng trắng)
    string = re.sub(r'[-()\\"#/@;:<>{}`+=~|.!?,%/]', "", string)
    string = re.sub(r"--|\n", " ", string)

    #   Thay số → token
    string = re.sub(r"\d+", "number", string)

    #   Xóa pattern Facebook
    for pattern in FACEBOOK_PATTERNS:
        string = re.sub(pattern, "", string)

    #  Xóa ký tự thừa
    string = string.replace('"', " ").replace("️", "").replace("🏻", "")

    #  Chuẩn hóa khoảng trắng
    string = re.sub(r" {2,}", " ", string).strip()

    return string

#Kiểm tra nhanh
samples = [
    "Thầy dạy rất nhiệt tình 😊👍 nhưng bài tập nhiều quá 😭",
    "môn này ổnnnnn ạ, không có gì đặc biệt 🙄",
    "GHÉT MÔN NÀY VÃI!!! 😡😠 phí tiền phí thời gian",
]
print("🔬 Kiểm tra hàm Preprocess:")
print("-" * 60)
for s in samples:
    print(f"  IN : {s}")
    print(f"  OUT: {Preprocess(s)}")
    print()


📋 Tổng số emoji trong bảng: 132
   😊 Positive : 81
   😢 Negative : 37
   😐 Neutral  : 14
🔬 Kiểm tra hàm Preprocess:
------------------------------------------------------------
  IN : Thầy dạy rất nhiệt tình 😊👍 nhưng bài tập nhiều quá 😭
  OUT: thầy dạy rất nhiệt tình positive positive nhưng bài tập nhiều quá negative

  IN : môn này ổnnnnn ạ, không có gì đặc biệt 🙄
  OUT: môn này ổn ạ không có gì đặc biệt neutral

  IN : GHÉT MÔN NÀY VÃI!!! 😡😠 phí tiền phí thời gian
  OUT: ghét môn này vãi negative negative phí tiền phí thời gian



In [ ]:
def replace_teencode(text, dictionary):
    """Thay thế teencode theo từ điển (so khớp từng token)."""
    tokens = text.split(" ")
    replaced = [dictionary.get(token, token) for token in tokens]
    return " ".join(replaced).strip()

In [ ]:
X = df["sentence"]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Reset index để tránh lỗi khi dùng .values[i]
X_train = X_train.reset_index(drop=True)
X_valid   = X_valid.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_valid   = y_valid.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print(f"\nSplit sizes → train: {len(X_train)}, dev: {len(X_valid)}, test: {len(X_test)}")


Split sizes → train: 22472, dev: 2809, test: 2810


In [ ]:
print("\nPreprocessing...")

for i in range(len(X_train)):
    X_train.values[i] = Preprocess(X_train.values[i])
for i in range(len(X_valid)):
    X_valid.values[i]   = Preprocess(X_valid.values[i])
for i in range(len(X_test)):
    X_test.values[i]  = Preprocess(X_test.values[i])

print("Replacing teencode...")

for i in range(len(X_train)):
    X_train.values[i] = replace_teencode(X_train.values[i], dictionary)
for i in range(len(X_valid)):
    X_valid.values[i]   = replace_teencode(X_valid.values[i], dictionary)
for i in range(len(X_test)):
    X_test.values[i]  = replace_teencode(X_test.values[i], dictionary)

print("Done!")
print("\nSample after preprocessing:")
for i in range(3):
    print(f"  [{y_train[i]}] {X_train[i]}")


Preprocessing...
Replacing teencode...
Done!

Sample after preprocessing:
  [neutral] ai học bài thi TOEIC ở mới hoa chưa ạ cho mình xin đánh giá với
  [neutral] thắc mắc ngay tại lớp những cái mình không hiểu
  [positive] giảng viên giúp sinh viên xác định rõ ràng mục đích học tập


In [ ]:
all_results = {}

In [ ]:
import os
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Dùng đường dẫn AN TOÀN (không dấu để tránh lỗi)
base_path = "/content/drive/MyDrive/Nguyen_Thi_Suong_Anh/Do_an_co_so/DACS/data"

# 3. Tạo thư mục dataset trong Drive
folder_path = os.path.join(base_path, "dataset")
os.makedirs(folder_path, exist_ok=True)

# 4. Tạo DataFrame
train_df = pd.DataFrame({'sentence': X_train, 'sentiment': y_train})
valid_df = pd.DataFrame({'sentence': X_valid, 'sentiment': y_valid})
test_df  = pd.DataFrame({'sentence': X_test,  'sentiment': y_test})

# 5. Lưu trực tiếp vào Drive
train_path = os.path.join(folder_path, "train.csv")
valid_path = os.path.join(folder_path, "valid.csv")
test_path  = os.path.join(folder_path, "test.csv")

train_df.to_csv(train_path, index=False, encoding="utf-8-sig")
valid_df.to_csv(valid_path, index=False, encoding="utf-8-sig")
test_df.to_csv(test_path, index=False, encoding="utf-8-sig")

# 6. In kết quả kiểm tra
print("Đã lưu vào Google Drive:")
print(train_path)
print(valid_path)
print(test_path)

print("\nDanh sách file trong folder:")
print(os.listdir(folder_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã lưu vào Google Drive:
/content/drive/MyDrive/Nguyen_Thi_Suong_Anh/Do_an_co_so/DACS/data/dataset/train.csv
/content/drive/MyDrive/Nguyen_Thi_Suong_Anh/Do_an_co_so/DACS/data/dataset/valid.csv
/content/drive/MyDrive/Nguyen_Thi_Suong_Anh/Do_an_co_so/DACS/data/dataset/test.csv

Danh sách file trong folder:
['train.csv', 'valid.csv', 'test.csv']


In [ ]:
import webbrowser
webbrowser.open("https://drive.google.com/drive/my-drive")

False